# 01 Data and Exploratory Data Analysis

## Objective

This notebook examines the return, volatility and cross-sectional 
dependence characteristics of the cryptocurrency universe.

The main objective is to determine whether the assets exhibit substantial
common market exposure before constructing cross-sectional trading signals.

Development sample: 2020–2023  
Out-of-sample evaluation: 2024–2025

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load cleaned hourly crypto data
df = pd.read_csv(
    "crypto_hourly_clean_2020_2025.csv",
    parse_dates=["open_time"]
)

df = (
    df
    .sort_values(["symbol", "open_time"])
    .reset_index(drop=True)
)

print("Shape:", df.shape)
print("Period:", df["open_time"].min(), "to", df["open_time"].max())
print("Assets:", df["symbol"].nunique())
print("Symbols:", sorted(df["symbol"].unique()))

display(df.head())

Shape: (420616, 8)
Period: 2020-01-01 00:00:00+00:00 to 2025-12-31 23:00:00+00:00
Assets: 8
Symbols: ['ADAUSDT', 'BNBUSDT', 'BTCUSDT', 'DOGEUSDT', 'ETHUSDT', 'LINKUSDT', 'LTCUSDT', 'XRPUSDT']


,symbol,open_time,open,high,low,close,volume,ret_1h
0,ADAUSDT,2020-01-01 00:00:00+00:00,0.03285,0.03285,0.03270,0.03278,1166000.9,-0.002131
1,ADAUSDT,2020-01-01 01:00:00+00:00,0.03277,0.03303,0.03276,0.03299,1560751.8,0.006406
2,ADAUSDT,2020-01-01 02:00:00+00:00,0.03299,0.03320,0.03298,0.03317,1091975.1,0.005456
3,ADAUSDT,2020-01-01 03:00:00+00:00,0.03319,0.03319,0.03297,0.03303,739364.9,-0.004221
4,ADAUSDT,2020-01-01 04:00:00+00:00,0.03301,0.03305,0.03290,0.03299,1350480.0,-0.001211


In [3]:
# step 3: 读取development sample
dev = df[
    (df["open_time"] >= "2020-01-01") &
    (df["open_time"] < "2024-01-01")
].copy()

print("Development period:")
print(dev["open_time"].min(), "to", dev["open_time"].max())
print("Rows:", len(dev))
print("Assets:", dev["symbol"].nunique())

Development period:
2020-01-01 00:00:00+00:00 to 2023-12-31 23:00:00+00:00
Rows: 280264
Assets: 8


In [4]:
# step 4:
display(
    dev["ret_1h"]
    .describe(
        percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
    )
)


count    280144.000000
mean          0.000126
std           0.011291
min          -0.259478
1%           -0.030964
5%           -0.014718
25%          -0.003798
50%           0.000049
75%           0.003997
95%           0.014852
99%           0.031970
max           0.700192
Name: ret_1h, dtype: float64

In [5]:
# Step 5：return characteristics
asset_stats = (
    dev.groupby("symbol")["ret_1h"]
    .agg(["mean", "std", "min", "max", "skew"])
)

asset_stats["kurtosis"] = (
    dev.groupby("symbol")["ret_1h"]
    .apply(pd.Series.kurt)
)

display(asset_stats.round(4))

,mean,std,min,max,skew,kurtosis
symbol,,,,,,
ADAUSDT,0.0001,0.0115,-0.2154,0.1504,-0.1395,18.4153
BNBUSDT,0.0001,0.0099,-0.2027,0.1480,-0.5748,33.3236
BTCUSDT,0.0001,0.0073,-0.1821,0.1738,-0.4634,49.1001
DOGEUSDT,0.0002,0.0155,-0.2539,0.7002,5.5868,211.6200
ETHUSDT,0.0001,0.0092,-0.2087,0.1501,-0.6250,24.6358
LINKUSDT,0.0001,0.0125,-0.2305,0.1770,-0.2935,17.8655
LTCUSDT,0.0001,0.0107,-0.2595,0.2494,-0.4863,38.0240
XRPUSDT,0.0001,0.0119,-0.2014,0.2775,0.7088,47.6461


In [6]:
# Step 6：Correlation
return_panel = (
    dev
    .pivot(
        index="open_time",
        columns="symbol",
        values="ret_1h"
    )
    .sort_index()
)

display(return_panel.head())

symbol,ADAUSDT,BNBUSDT,BTCUSDT,DOGEUSDT,ETHUSDT,LINKUSDT,LTCUSDT,XRPUSDT
open_time,,,,,,,,
2020-01-01 00:00:00+00:00,-0.002131,-0.001312,-0.002531,-0.002583,-0.002245,-0.001980,-0.000484,-0.002436
2020-01-01 01:00:00+00:00,0.006406,0.007402,0.005469,0.005577,0.013735,0.010942,0.008236,0.006390
2020-01-01 02:00:00+00:00,0.005456,0.003500,0.003683,0.004803,0.001607,0.009702,0.005526,0.002426
2020-01-01 03:00:00+00:00,-0.004221,-0.002224,-0.002463,-0.009610,-0.004968,-0.003999,-0.007646,-0.001081
2020-01-01 04:00:00+00:00,-0.001211,-0.001795,-0.001071,0.001642,0.000000,0.002008,0.001445,-0.001753


In [7]:
corr_matrix = return_panel.corr()

display(corr_matrix.round(2))

symbol,ADAUSDT,BNBUSDT,BTCUSDT,DOGEUSDT,ETHUSDT,LINKUSDT,LTCUSDT,XRPUSDT
symbol,,,,,,,,
ADAUSDT,1.00,0.66,0.68,0.44,0.72,0.71,0.70,0.61
BNBUSDT,0.66,1.00,0.71,0.42,0.74,0.69,0.69,0.58
BTCUSDT,0.68,0.71,1.00,0.47,0.85,0.71,0.76,0.61
DOGEUSDT,0.44,0.42,0.47,1.00,0.47,0.44,0.46,0.39
ETHUSDT,0.72,0.74,0.85,0.47,1.00,0.77,0.79,0.63
LINKUSDT,0.71,0.69,0.71,0.44,0.77,1.00,0.72,0.59
LTCUSDT,0.70,0.69,0.76,0.46,0.79,0.72,1.00,0.64
XRPUSDT,0.61,0.58,0.61,0.39,0.63,0.59,0.64,1.00


In [8]:
n = len(corr_matrix)

avg_pairwise_corr = (
    (corr_matrix.values.sum() - n)
    / (n * (n - 1))
)

print(
    "Average pairwise correlation:",
    round(avg_pairwise_corr, 3)
)

Average pairwise correlation: 0.631


- During the EDA, I found strong co-movement across cryptocurrencies, 
- with an average pairwise hourly-return correlation of around 0.63. 
- This suggested that a substantial part of individual crypto returns was driven by a common market factor.
- So rather than directly ranking raw returns, I decided to remove the common market component 
- and investigate whether asset-specific residual returns contained predictive information.”

## EDA Conclusion

- Cryptocurrency returns exhibit substantial cross-sectional co-movement.
- The average pairwise hourly-return correlation is approximately 0.63 in the 2020–2023 development sample.
- Major assets such as BTC and ETH show particularly strong correlation.
- This suggests that raw asset returns contain a substantial common crypto-market component.
- The next stage therefore removes common market exposure and investigates predictive signals in residual returns.